In [45]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

Menggunakan device: cpu


In [46]:
# Jalur relatif dari folder 'notebooks/' ke direktori utama
BASE_DIR = os.path.abspath("..")

# Mengakomodasi folder data/test/test/
TEST_DIR = os.path.join(BASE_DIR, "data", "test", "test")
SAMPLE_SUB_PATH = os.path.join(BASE_DIR, "data", "sample_submission.csv")
MODEL_PATH = os.path.join(BASE_DIR, "models", "best_model.pth")
OUTPUT_DIR = os.path.join(BASE_DIR, "submissions")
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "submission.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Path Test Gambar : {TEST_DIR}")
print(f"Path Best Model  : {MODEL_PATH}")

Path Test Gambar : d:\Semester 5\SD4504_SML\Document-Image-Classification\data\test\test
Path Best Model  : d:\Semester 5\SD4504_SML\Document-Image-Classification\models\best_model.pth


In [47]:
class DocumentTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Ambil nama file gambar dari kolom pertama (misal: 'id' atau 'image')
        img_name = str(self.df.iloc[idx, 0])
        
        # Pastikan ektensi .tif ada
        if not img_name.endswith('.tif'):
            img_name += '.tif'
            
        img_path = os.path.join(self.img_dir, img_name)
        
        # Buka format .tif dan konversi ke RGB
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, img_name

# Transformasi input gambar
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

sub_df = pd.read_csv(SAMPLE_SUB_PATH)

test_dataset = DocumentTestDataset(sub_df, TEST_DIR, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Inisialisasi arsitektur EfficientNet-B0
model = models.efficientnet_b0(weights=None)

# Sesuaikan num_classes dengan jumlah kelas saat training (misal: 16)
num_classes = 16  
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

# Load weights dari models/best_model.pth
checkpoint = torch.load(MODEL_PATH, map_location=device)

if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])
elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()
print("Model berhasil dimuat dari best_model.pth!")

Model berhasil dimuat dari best_model.pth!


In [ ]:
all_preds = []

with torch.no_grad():
    for images, _ in tqdm(test_loader, desc="Menjalankan Inferensi"):
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)

# Isi hasil prediksi ke kolom label target (kolom ke-2 di sample_submission.csv)
target_col = sub_df.columns[1]
sub_df[target_col] = all_preds

# Simpan CSV ke direktori submissions/
sub_df.to_csv(OUTPUT_FILE, index=False)
print(f"File submission berhasil dibuat di: {OUTPUT_FILE}")
sub_df.head()

Menjalankan Inferensi:   0%|          | 0/250 [00:00<?, ?it/s]